In [ ]:
import datasets
from uuid import uuid4

# Transforming EmpatheticIntents

In [ ]:
dataset = datasets.load_dataset("psytechlab/EmpatheticIntents-ru")

In [ ]:
ACTOR = {
    'none': "никто",
    'speaker': "рассказщик",
    'listener': 'слушатель'
}

ALL_LABELS = {
  "surprised": "удивлённый",  
  "sentimental": "сентиментальный",  
  "content": "довольный",  
  "trusting": "доверчивый",  
  "disappointed": "разочарованный",  
  "questioning": "сомневающийся",  
  "sad": "грустный",  
  "agreeing": "соглашающийся",  
  "joyful": "радостный",  
  "encouraging": "ободряющий",  
  "annoyed": "раздражённый",  
  "terrified": "напуганный",  
  "consoling": "утешающий",  
  "wishing": "желающий",  
  "acknowledging": "признающий",  
  "neutral": "нейтральный",  
  "nostalgic": "ностальгирующий",  
  "lonely": "одинокий",  
  "anxious": "тревожный",  
  "embarrassed": "смущённый",  
  "proud": "гордый",  
  "guilty": "виноватый",  
  "prepared": "готовый",  
  "furious": "разъярённый",  
  "caring": "заботливый",  
  "confident": "уверенный",  
  "jealous": "ревнивый",  
  "hopeful": "надеющийся",  
  "impressed": "впечатлённый",  
  "disgusted": "отвращённый",  
  "grateful": "благодарный",  
  "excited": "взволнованный",  
  "faithful": "верный",  
  "apprehensive": "опасающийся",  
  "suggesting": "предлагающий",  
  "angry": "злой",  
  "devastated": "опустошённый",  
  "afraid": "испуганный",  
  "anticipating": "ожидающий",  
  "sympathizing": "сочувствующий",  
  "ashamed": "пристыженный"  
}

In [ ]:
ALL_LABELS = {y:x for x,y in ALL_LABELS.items()}
ACTOR = {y:x for x,y in ACTOR.items()}

In [ ]:
examples = []
for ex in dataset["train"]:
    uters = []
    for i in ex["dialog"]:
        u = {
            "role": ACTOR[i["actor"]],
            "text": {
                "rus": i["text"],
                "eng": i["text_eng"]
            },
            "annotation": {
                "emotion": ALL_LABELS[i["label"]]
            }
        }
        uters.append(u)
    ex_ = {
        "id": ex["dialog_id"],
        "annotation": {
            "emotion": ex["emotion"]
        },
        "meta": {
            "situation" : {
                "rus": ex["situation_rus"],
                "eng": ex["situation_eng"]
            }
        },
        "utterances": uters
    }
    examples.append(ex_)

In [ ]:
new_dataset = datasets.Dataset.from_list(examples)

In [ ]:
new_dataset = datasets.DatasetDict({"train": new_dataset})

In [ ]:
new_dataset.push_to_hub("psytechlab/EmpatheticIntents-ru")

In [ ]:
dataset = datasets.load_dataset("psytechlab/ESConv-ru")

In [ ]:
EXPERIENCE_TYPES = {
    'Current Experience': 'Текущий опыт',
    'Previous Experience': 'Предыдущий опыт'
}

EMOTION_TYPES = {
    'anger': 'гнев',
    'anxiety': 'тревога',
    'depression': 'депрессия',
    'disgust': 'отвращение',
    'fear': 'страх',
    'guilt': 'вина',
    'jealousy': 'ревность',
    'nervousness': 'нервозность',
    'pain': 'боль',
    'sadness': 'грусть',
    'shame': 'стыд'
}

PROBLEM_TYPES = {
    'Alcohol Abuse': 'Злоупотребление алкоголем',
    'Appearance Anxiety': 'Тревога из-за внешности',
    'Issues with Children': 'Проблемы с детьми',
    'Issues with Parents': 'Проблемы с родителями',
    'Procrastination': 'Прокрастинация',
    'School Bullying': 'Школьная травля',
    'Sleep Problems': 'Проблемы со сном',
    'academic pressure': 'Учебное давление',
    'breakup with partner': 'Расставание с партнёром',
    'conflict with parents': 'Конфликт с родителями',
    'job crisis': 'Кризис на работе',
    'ongoing depression': 'Затяжная депрессия',
    'problems with friends': 'Проблемы с друзьями'
}

STRATEGY = {
        'Affirmation and Reassurance': 'Подтверждение и заверение',
        'Information': 'Информирование',
        'Others': 'Другое',
        'Providing Suggestions': 'Предоставление советов',
        'Question': 'Вопрос',
        'Reflection of feelings': 'Отражение чувств',
        'Restatement or Paraphrasing': 'Перефразирование',
        'Self-disclosure': 'Самораскрытие'
    }

In [ ]:
EXPERIENCE_TYPES = {y:x for x,y in EXPERIENCE_TYPES.items()}
EMOTION_TYPES = {y:x for x,y in EMOTION_TYPES.items()}
PROBLEM_TYPES = {y:x for x,y in PROBLEM_TYPES.items()}
STRATEGY = {y:x for x,y in STRATEGY.items()}

In [ ]:
ANNOT = {
    "strategy" : STRATEGY,
    "feedback" : {
        "0":"0",
        "1":"1",
        "2":"2",
        "3":"3",
        "4":"4",
        "5":"5"
    }
}
role_mapping = {"supporter": "listener",
"seeker": "speaker"}

In [ ]:
examples = []
for ex in dataset["train"]:
    uters = []
    for i in ex["dialog"]:
        for k,v in i["annotation"].items():
            i["annotation"][k] = ANNOT[k][v] if v is not None else v
        u = {
            "role": role_mapping[i["speaker"]],
            "text": {
                "rus": i["content"].strip(),
                "eng": i["content_eng"].strip()
            },
            "annotation": i["annotation"]
                
        }
        uters.append(u)
    ex_ = {
        "id": uuid4().hex,
        "utterances": uters,
        "annotation": {
            "experience_type": EXPERIENCE_TYPES[ex["experience_type"]],
            "emotion": EMOTION_TYPES[ex["emotion_type"]],
            "problem_type": PROBLEM_TYPES[ex["problem_type"]]
        },
        "meta":{
            "situation" : {
                "rus": ex["situation"],
                "eng": ex["situation_eng"]
            },
            "seeker_question1": ex["seeker_question1"],
            "seeker_question2": ex["seeker_question2"],
            "supporter_question1": ex["supporter_question1"],
            "supporter_question2": ex["supporter_question2"],
            "survey_score": ex["survey_score"]
        }
    }
    examples.append(ex_)

In [ ]:
new_dataset = datasets.DatasetDict({"train": datasets.Dataset.from_list(examples)})
new_dataset.push_to_hub("psytechlab/ESConv-ru")

In [ ]:
dataset = datasets.load_dataset("psytechlab/epitome-reddit-ru")

In [ ]:
new_dataset = {}
for k,v in dataset.items():
    examples = []
    for ex in v:
        dialog = {
             "id": f"{ex['sp_id']}:{ex['rp_id']}",
            "utterances" : [{
                "role": "speaker",
                "text": {
                    "rus": ex["seeker_post"],
                },
                    
            },
            {
                "role": "listener",
                "text": {
                    "rus": ex["response_post"],
                },
                "annotation": {
                    "intensity": ex["level"],
                    "rationales": ex["rationales"]
                }
            }]
        }
        examples.append(dialog)
    new_dataset[k] = datasets.Dataset.from_list(examples)

In [ ]:
new_dataset = datasets.DatasetDict(new_dataset)

In [ ]:
new_dataset.push_to_hub("psytechlab/epitome-reddit-ru")